In [0]:
%pip install -q langgraph langchain-google-genai python-dotenv

In [0]:
# Reinicia o Python para aplicar os pacotes instalados (célula separada da instalação e dos imports)
dbutils.library.restartPython()

In [0]:
import glob
import os
from pathlib import Path
from dotenv import load_dotenv

# Pasta do projeto localizada dinamicamente — nenhum usuário/e-mail hardcoded no código
_caminhos = glob.glob('/Workspace/Users/*/GenAI-Financial')
assert _caminhos, 'Pasta GenAI-Financial não encontrada em /Workspace/Users/'
assert len(_caminhos) == 1, f'Esperava 1 pasta GenAI-Financial, encontrei: {_caminhos}'
PROJETO = Path(_caminhos[0])
PROMPTS_DIR = PROJETO / 'prompts'
REPORTS_DIR = PROJETO / 'reports'
REPORTS_DIR.mkdir(exist_ok=True)

# Credenciais fora do versionamento (.env já coberto pelo .gitignore)
load_dotenv(PROJETO / '.env')

# Rastreio das execuções no LangSmith (usa LANGCHAIN_API_KEY do .env)
os.environ['LANGCHAIN_TRACING_V2'] = 'true'

# Prompts versionados em arquivo: nova versão = novo arquivo (_v2, _v3...)
AGENTE1_PROMPT = PROMPTS_DIR / 'agente1_analise_v1.md'
AGENTE2_PROMPT = PROMPTS_DIR / 'agente2_report_v1.md'

def carregar_prompt(caminho):
    with open(caminho) as arquivo:
        return arquivo.read()

print('Agente 1:', AGENTE1_PROMPT.name, '—', len(carregar_prompt(AGENTE1_PROMPT)), 'caracteres')
print('Agente 2:', AGENTE2_PROMPT.name, '—', len(carregar_prompt(AGENTE2_PROMPT)), 'caracteres')

In [0]:
import json
import pandas as pd

df_gold = (
    spark
    .table('db_analytics.gold_bitcoin')
    .orderBy('semana_inicio')
    .toPandas()
)

# Normaliza datas: o Spark pode devolver datetime.date em colunas object
# (sem isso, as chamadas .date() abaixo falham com AttributeError)
df_gold['semana_inicio'] = pd.to_datetime(df_gold['semana_inicio'])
df_gold['semana_fim'] = pd.to_datetime(df_gold['semana_fim'])

def registros(df):
    df = df.copy()
    df['semana_inicio'] = df['semana_inicio'].astype(str)
    df['semana_fim'] = df['semana_fim'].astype(str)
    return json.loads(df.to_json(orient='records'))

def resumo(df, nome):
    return {
        'periodo': nome,
        'variacao_pct_media': round(df['variacao_pct'].mean(), 2),
        'fechamento_medio': round(df['fechamento'].mean(), 2),
        'volume_medio': round(df['volume_medio'].mean(), 0),
    }

recentes = df_gold.tail(4)
anteriores = df_gold.tail(8).head(4)
melhor = df_gold.loc[df_gold['variacao_pct'].idxmax()]
pior = df_gold.loc[df_gold['variacao_pct'].idxmin()]

dados_para_agente = json.dumps({
    'fonte': 'db_analytics.gold_bitcoin — agregação semanal do BTC-USD',
    'historico_completo': {
        'semanas': int(len(df_gold)),
        'primeira_semana': str(df_gold['semana_inicio'].iloc[0].date()),
        'ultima_semana': str(df_gold['semana_inicio'].iloc[-1].date()),
        'resumo': resumo(df_gold, 'histórico completo'),
    },
    'comparativo': [
        resumo(anteriores, 'semanas 8 a 4 mais recentes'),
        resumo(recentes, 'últimas 4 semanas'),
    ],
    'semanas_em_destaque': {
        'melhor': {'semana_inicio': str(melhor['semana_inicio'].date()), 'variacao_pct': round(melhor['variacao_pct'], 2)},
        'pior': {'semana_inicio': str(pior['semana_inicio'].date()), 'variacao_pct': round(pior['variacao_pct'], 2)},
    },
    'ultimas_8_semanas': registros(df_gold.tail(8)),
    'semana_mais_recente': registros(df_gold.tail(1)),
}, ensure_ascii=False)

print('Semanas analisadas:', len(df_gold))
print('Última semana disponível:', str(df_gold['semana_inicio'].iloc[-1].date()))

In [0]:
from typing_extensions import TypedDict
from langgraph.graph import StateGraph, START, END
from langchain_google_genai import ChatGoogleGenerativeAI

llm = ChatGoogleGenerativeAI(model='gemini-3.8-flash', temperature=0.2, timeout=600)

def extrair_texto(resposta):
    """Normaliza resposta.content para str (modelos recentes retornam lista de partes)."""
    conteudo = resposta.content
    if isinstance(conteudo, list):
        return ''.join(p.get('text', '') if isinstance(p, dict) else str(p) for p in conteudo)
    return conteudo

class EstadoAgentes(TypedDict):
    dados: str           # dados semanais em JSON (entrada)
    analise: str         # Agente 1 — análise financeira comparativa
    relatorio_html: str  # Agente 2 — report em HTML

def agente_analise_financeira(estado: EstadoAgentes) -> dict:
    """Agente 1 — análise financeira comparativa (prompt versionado em arquivo)."""
    resposta = llm.invoke([
        ('system', carregar_prompt(AGENTE1_PROMPT)),
        ('human', estado['dados']),
    ])
    return {'analise': extrair_texto(resposta)}

def agente_gerador_report(estado: EstadoAgentes) -> dict:
    """Agente 2 — geração do report em HTML (prompt versionado em arquivo)."""
    mensagem = (
        'ANÁLISE DO AGENTE 1:\n\n' + estado['analise']
        + '\n\nDADOS SEMANAIS (JSON):\n' + estado['dados']
    )
    resposta = llm.invoke([
        ('system', carregar_prompt(AGENTE2_PROMPT)),
        ('human', mensagem),
    ])
    return {'relatorio_html': extrair_texto(resposta)}

grafo = StateGraph(EstadoAgentes)
grafo.add_node('agente_analise', agente_analise_financeira)
grafo.add_node('agente_report', agente_gerador_report)
grafo.add_edge(START, 'agente_analise')
grafo.add_edge('agente_analise', 'agente_report')
grafo.add_edge('agente_report', END)

pipeline_agentes = grafo.compile()

print(pipeline_agentes.get_graph().draw_mermaid())

In [0]:
from datetime import datetime

resultado = pipeline_agentes.invoke({'dados': dados_para_agente})

# Agente 1 — análise financeira comparativa
print(resultado['analise'])
print('=' * 70)

# Agente 2 — report HTML (remove cercas de código, caso existam)
relatorio_html = resultado['relatorio_html'].strip()
if relatorio_html.startswith('```'):
    relatorio_html = relatorio_html.strip('`').removeprefix('html').strip()

arquivo_relatorio = REPORTS_DIR / f"relatorio_bitcoin_{datetime.now():%Y%m%d_%H%M}.html"
with open(arquivo_relatorio, 'w') as arquivo:
    arquivo.write(relatorio_html)

print(f'Relatório salvo em: {arquivo_relatorio}')
displayHTML(relatorio_html)